In [1]:
!pip install gensim

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [31]:
import pandas as pd

fake_df = pd.read_csv('/content/drive/MyDrive/Fake news detection /Fake.csv')
true_df = pd.read_csv('/content/drive/MyDrive/Fake news detection /True.csv')

fake_df['label'] = 0
true_df['label'] = 1


df = pd.concat([fake_df, true_df], ignore_index=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.head())
print(df['label'].value_counts())


                                               title  \
0  Ben Stein Calls Out 9th Circuit Court: Committ...   
1  Trump drops Steve Bannon from National Securit...   
2  Puerto Rico expects U.S. to lift Jones Act shi...   
3   OOPS: Trump Just Accidentally Confirmed He Le...   
4  Donald Trump heads for Scotland to reopen a go...   

                                                text       subject  \
0  21st Century Wire says Ben Stein, reputable pr...       US_News   
1  WASHINGTON (Reuters) - U.S. President Donald T...  politicsNews   
2  (Reuters) - Puerto Rico Governor Ricardo Rosse...  politicsNews   
3  On Monday, Donald Trump once again embarrassed...          News   
4  GLASGOW, Scotland (Reuters) - Most U.S. presid...  politicsNews   

                  date  label  
0    February 13, 2017      0  
1       April 5, 2017       1  
2  September 27, 2017       1  
3         May 22, 2017      0  
4       June 24, 2016       1  
label
0    23481
1    21417
Name: count, dtype: in

In [32]:
df.isna().sum()

,0
title,0
text,0
subject,0
date,0
label,0


In [34]:
df['Processed_data'] = df['title'] + " " + df['text']

In [35]:
#For removing any special characters
import re
#For removing stopwords
import nltk
nltk.download('stopwords',quiet = True)
#Stemming the text
from nltk.stem import PorterStemmer
#Importing NumPy
import numpy as np
# Ignore warnings
import warnings
warnings.filterwarnings('ignore', category = FutureWarning)

In [36]:
df['Processed_data'] = df['Processed_data'].str.lower()
df.head()

,title,text,subject,date,label,Processed_data
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0,ben stein calls out 9th circuit court: committ...
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1,trump drops steve bannon from national securit...
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1,puerto rico expects u.s. to lift jones act shi...
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0,oops: trump just accidentally confirmed he le...
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1,donald trump heads for scotland to reopen a go...


In [37]:
#Removing any kind of special characters that might be present in out text by defining a function
def remove_special_chars(text):
    return re.sub(r'[^a-zA-Z\s]', '', text)

df['Processed_data'] = df['Processed_data'].apply(remove_special_chars)

df.head()

,title,text,subject,date,label,Processed_data
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0,ben stein calls out th circuit court committed...
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1,trump drops steve bannon from national securit...
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1,puerto rico expects us to lift jones act shipp...
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0,oops trump just accidentally confirmed he lea...
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1,donald trump heads for scotland to reopen a go...


In [38]:
#Removing stopwords from the text

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df['Processed_data'] = df['Processed_data'].apply(remove_stopwords)
df.head()

,title,text,subject,date,label,Processed_data
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0,ben stein calls th circuit court committed cou...
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1,trump drops steve bannon national security cou...
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1,puerto rico expects us lift jones act shipping...
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0,oops trump accidentally confirmed leaked israe...
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1,donald trump heads scotland reopen golf resort...


In [39]:
# Stemming the data
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
def stem_words(text):
    words = text.split()
    words = [stemmer.stem(word) for word in words]
    return " ".join(words)

df['Processed_data'] = df['Processed_data'].apply(stem_words)
display(df.head())

,title,text,subject,date,label,Processed_data
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0,ben stein call th circuit court commit coup dt...
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1,trump drop steve bannon nation secur council w...
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1,puerto rico expect us lift jone act ship restr...
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0,oop trump accident confirm leak isra intellig ...
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1,donald trump head scotland reopen golf resort ...


In [40]:
import gensim
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [41]:
import nltk
nltk.download('punkt',quiet=True)
nltk.download('punkt_tab',quiet=True)
news = []
for doc in df['Processed_data']:
  raw_sent = sent_tokenize(doc)
  for sent in raw_sent:
    news.append(simple_preprocess(sent))

In [42]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [43]:
model.build_vocab(news)

In [44]:
model.train(news,total_examples=model.corpus_count,epochs=model.epochs)

(51198382, 53259955)

In [45]:
len(model.wv.index_to_key)

93243

In [49]:
def document_vector(doc):
  # Filter out words not in the model's vocabulary
  doc = [word for word in doc.split() if word in model.wv.index_to_key]
  if not doc:
    # Return a zero vector if the document is empty after filtering
    return np.zeros(model.vector_size)
  return np.mean(model.wv[doc], axis=0)

In [47]:
document_vector(df['Processed_data'].values[0])

array([ 0.15568683,  0.6657084 ,  0.97622937, -0.23029806,  0.05840553,
       -0.17769656, -0.65903306,  0.2708519 ,  0.53068614, -0.2551483 ,
        0.6571271 , -0.01884572,  0.47352475,  0.19435483,  0.0279811 ,
        0.21487308,  0.15451345,  0.3771892 , -0.38815767, -0.0710447 ,
        0.3559348 ,  0.53109735,  0.68452704, -0.864063  ,  0.06126153,
       -0.14186832,  0.06222832,  0.68886524, -0.12275993, -0.29895896,
       -0.1785477 , -0.1997771 , -0.00454503,  0.03631167, -0.9589778 ,
       -0.35842326, -0.8986787 , -0.00948848, -0.4281704 , -0.28071183,
       -0.38558203, -0.59621775, -0.31575453, -0.01847415,  0.31076145,
        0.24655981, -0.04855142, -0.29394996, -0.28382254,  0.1940245 ,
        0.3938159 ,  0.00260133,  0.50876826,  0.15823255,  0.49496144,
        0.65241367, -0.98998696, -0.4032338 ,  0.70087385, -0.6253346 ,
       -0.12932526,  0.00245331, -0.750195  ,  0.21178845,  1.2704017 ,
       -0.36160648,  0.58308417,  0.11290909, -0.3961921 ,  0.07

In [50]:
from tqdm import tqdm
x = []
for doc in tqdm(df['Processed_data'].values):
  x.append(document_vector(doc))

100%|██████████| 44898/44898 [14:15<00:00, 52.49it/s]


In [51]:
x = np.array(x)

In [52]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(df['label'])

In [53]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [54]:
rf = RandomForestClassifier()
rf.fit(x_train,y_train)
y_pred = rf.predict(x_test)
accuracy_score(y_test,y_pred)

0.9687082405345212

In [59]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

texts = df["Processed_data"].astype(str)
labels = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words='english',
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF feature matrix: {X_train_tfidf.shape}")
rftf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)
rftf.fit(X_train_tfidf, y_train)
y_pred = rftf.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred)*100, 2), "%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


TF-IDF feature matrix: (35918, 10000)
Accuracy: 99.74 %

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      4696
           1       1.00      1.00      1.00      4284

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980


Confusion Matrix:
 [[4680   16]
 [   7 4277]]


In [61]:
import joblib

joblib.dump(rftf, "RandomForest.pkl")
joblib.dump(tfidf, "Vectorizer.pkl")
model.save("word2vec.model")